In [14]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

In [15]:
# project directory
abspath = os.path.abspath('')
project_dir = str(Path(abspath).parents[0])

# sub-directories
data_raw = os.path.join(project_dir, "data", "raw")
data_interim = os.path.join(project_dir, "data", "interim")
data_external = os.path.join(project_dir, "data", "external")
data_processed = os.path.join(project_dir, "data", "processed")

Define paths to lfs data

In [16]:
fpath_lfs = os.path.join(data_raw, "lfs", "eu")
main_data = r"Yearly Data\YearlyFiles_83_2019\_YearlyFiles"

Read geodata with NUTS regions

In [17]:
gdf = gpd.read_file(os.path.join(data_external, "geodata", "NUTS_RG_03M_2021_4326_LEVL_2", "NUTS_RG_03M_2021_4326_LEVL_2.shp"))
gdf.head()

,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,FID,geometry
0,AL01,2,AL,Veri,Veri,0.0,NaN,0,AL01,"POLYGON ((19.83100 42.46645, 19.84072 42.47881..."
1,AL02,2,AL,Qender,Qender,0.0,NaN,0,AL02,"POLYGON ((20.23617 41.34849, 20.29263 41.37277..."
2,DE24,2,DE,Oberfranken,Oberfranken,0.0,NaN,0,DE24,"POLYGON ((11.48157 50.43162, 11.48290 50.40161..."
3,IE05,2,IE,Southern,Southern,0.0,NaN,0,IE05,"MULTIPOLYGON (((-7.67408 52.78129, -7.61737 52..."
4,HU11,2,HU,Budapest,Budapest,0.0,NaN,0,HU11,"POLYGON ((18.93274 47.57117, 19.02713 47.58717..."


Read NACE section labels (n=21)

In [18]:
nace_sections = pd.read_csv(os.path.join(data_external, "classifications", "nace_rev2_1d_section_codes.csv"), delimiter=";")

Read ISCO group labels
- remove ISCO 4-digit
- pad others with zeros in the front to have 3 digits

In [19]:
df_isco = pd.read_csv(
    os.path.join(data_raw, "esco", "v1.1.0", "ISCOGroups_en.csv"),
    usecols=["code", "preferredLabel"]
)
df_isco = df_isco.loc[df_isco.code.astype(str).str.len() < 4].reset_index(drop=True)
df_isco.code = df_isco.code.astype(str)
df_isco.code = df_isco.code.str.pad(width=3, side="left", fillchar="0")
df_isco = df_isco.rename(columns={"preferredLabel": "ISCO3D_label"})

Read G/B/N classification
- aggregate from ESCO to ISCO 3-digit level

In [20]:
df_gbn = pd.read_csv(
    os.path.join(data_interim, "ESCO_ONET_METADATA_gbn.csv"),
    index_col=0,
    usecols=["preferred_label", "isco_level_3", "is_brown", "is_green", "is_neutral"]
)

df_gbn = df_gbn.rename(columns={"isco_level_3": "ISCO3D"})

df_gbn["type"] = df_gbn[["is_brown", "is_green", "is_neutral"]].idxmax(axis=1)
df_gbn = df_gbn[["ISCO3D", "type"]]

gbn_by_isco3 = df_gbn.groupby("ISCO3D").type.value_counts().to_frame("count").unstack().fillna(0)
gbn_by_isco3.columns = gbn_by_isco3.columns.droplevel()
gbn_by_isco3_shares = gbn_by_isco3.apply(lambda x: x / gbn_by_isco3.sum(axis=1), axis=0)
gbn_by_isco3_shares = gbn_by_isco3_shares.reset_index()
gbn_by_isco3_shares.ISCO3D = gbn_by_isco3_shares.ISCO3D.astype(str)

Combine ISCO labels with GBN classification

In [21]:
gbn_by_isco3_shares_merged = pd.merge(
    gbn_by_isco3_shares,
    df_isco,
    left_on="ISCO3D",
    right_on="code",
    how="left"
)
gbn_by_isco3_shares_merged = gbn_by_isco3_shares_merged.drop("code", axis=1)

#### Build dataset for all countries
Specify variables to read and their data types

In [22]:
# cols_nace-granularity_isco-granularity
cols = [
    "WSTATOR", "SEX", "NACE1D", "ISCO3D", "COUNTRYW", "REGIONW", "FTPT", "REFYEAR", 
    "COUNTRY", "REGION", "DEGURBA", "HHTYPE", "AGE", "ILOSTAT", "EDUC4WN", "HATLEV1D", "MAINSTAT", "INCDECIL", "COEFF"
]

# TODO: define NAN values globally or per column?
na_values = [9, 99, 999, 9999]

# define dtypes of variables
dtypes = {
    "WSTATOR": "category",
    "SEX": "category",
    "NACE1D": "category",
    "ISCO3D": "category",
    "COUNTRYW": "category",
    "REGIONW": "category",
    "FTPT": "category",
    "REFYEAR": "int",
    "COUNTRY": "category",
    "REGION": "category",
    "DEGURBA": "category",
    "HHTYPE": "category",
    "AGE": "int",
    "ILOSTAT": "category",
    "EDUC4WN": "category",
    "HATLEV1D": "category",
    "MAINSTAT": "category",
    "INCDECIL": "float"
}

# countries to process
countries = ["AT", "BE", "BG", "CH", "CY", "CZ", "DE", "DK", "EE", "ES", "FI", "FR", "GR", "HR", "HU", "IE", "IS", "IT", "LT", "LU", "LV", "MT", "NL", "NO", "PL", "PT", "RO", "SE", "SI", "SK", "UK"]

# year to process (currently only 1)
year = 2019

Stats for DE

In [23]:
# define fpath
country = "DE"
folder = "{}_YEAR_1998_onwards".format(country)
file = "{}{}_y.csv".format(country, year)
fpath_full = os.path.join(fpath_lfs, main_data, folder, file)

# read raw data
df = pd.read_csv(
    fpath_full,
    usecols=cols,
    na_values=na_values,
    dtype=dtypes,
    converters={"COEFF": lambda x: float(x) * 1000 if x != "" else np.nan}
)

df.COEFF.sum()

81831089.79750003

In [24]:
list_of_dfs = []
for country in countries:

    # define fpath
    folder = "{}_YEAR_1998_onwards".format(country)
    file = "{}{}_y.csv".format(country, year)
    fpath_full = os.path.join(fpath_lfs, main_data, folder, file)

    # read raw data
    df = pd.read_csv(
        fpath_full,
        usecols=cols,
        na_values=na_values,
        dtype=dtypes,
        converters={"COEFF": lambda x: float(x) * 1000 if x != "" else np.nan}
    )

    # filter based on conditions
    cond_is_working = df.WSTATOR.isin(["1", "2"])  # beschäftigt
    cond_private_household = df.HHTYPE.isin(["1"])  # privater wohnraum
    cond_has_isco_code = df.ISCO3D.notna()
    cond_not_inactive = df.ILOSTAT.isin(["1", "2"])  # inaktiv
    cond_in_country = df.COUNTRYW.isin([country])  # pendler
    cond_valid_region = df.REGIONW != "00"
    cond_age = df.AGE <= 77 # (77 is the center of the 75-79 age band)
    cond_military = ~df.ISCO3D.isin(["011"])  # military

    df_sub = df.loc[cond_is_working & cond_private_household & cond_not_inactive & cond_in_country & cond_valid_region & cond_has_isco_code & cond_military & cond_age]

    # remove categories that remain unused after filtering
    for col in df_sub.columns:
        if pd.api.types.is_categorical_dtype(df_sub[col]):
            df_sub.loc[:, col] = df_sub[col].cat.remove_unused_categories()

    # set NUTS code
    df_sub["NUTS_ID"] = df_sub["COUNTRYW"].astype(str) + df_sub["REGIONW"].astype(str)

    # append clean df
    list_of_dfs.append(df_sub)

C:\Users\fzaussinger\AppData\Local\Temp\ipykernel_15912\1070836824.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub.loc[:, col] = df_sub[col].cat.remove_unused_categories()
C:\Users\fzaussinger\AppData\Local\Temp\ipykernel_15912\1070836824.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub["NUTS_ID"] = df_sub["COUNTRYW"].astype(str) + df_sub["REGIONW"].astype(str)
C:\Users\fzaussinger\AppData\Local\Temp\ipykernel_15912\1070836824.py:33: SettingWithCopyWarning: 
A value is trying to be se

In [25]:
# Combine country-level data
df_merged = pd.concat(list_of_dfs, axis=0).reset_index(drop=True)
df_merged = df_merged.astype(dtypes)

# Add metadata to merged EU-wide data
df_merged_all_vars = pd.merge(
    df_merged,
    gbn_by_isco3_shares_merged,
    on="ISCO3D",
    how="left"
)

df_merged_all_vars = pd.merge(
    df_merged_all_vars,
    nace_sections,
    on="NACE1D",
    how="left"
)

# classify coal-related employment
cond_isco = df_merged_all_vars.ISCO3D_label.str.contains("mining|mine|miner", case=False, regex=True)
cond_nace = df_merged_all_vars.NACE1D.isin(["B", "D"])
df_merged_all_vars["mining"] = cond_isco & cond_nace

# Estimate employment by GBN
df_merged_all_vars["is_brown_abs"] = df_merged_all_vars.is_brown * df_merged_all_vars.COEFF
df_merged_all_vars["is_neutral_abs"] = df_merged_all_vars.is_neutral * df_merged_all_vars.COEFF
df_merged_all_vars["is_green_abs"] = df_merged_all_vars.is_green * df_merged_all_vars.COEFF

# Save to disk
df_merged_all_vars.to_csv(
    os.path.join(data_processed, "lfs", "eu_lfs_merged_2019.csv")
)